# DavnLoad_Save_Quotes



In [2]:
from pathlib import Path
import sys
import os

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"📂 Директория файла:                                      {file_dir}")

sub_project_dir = Path(file_dir).parent
print(f"📂 Директория СубПроекта:                     {sub_project_dir}")

project_dir                     = Path(file_dir).parent.parent.parent               # Переход в верхнюю директорию проекта (fc_to_mt5_migrations/own_platform)
print(f"📂 Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent.parent            # Переход на уровень выше (fc_to_mt5_migrations)
print(f"📂 Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")


# Функция для проверки существования директории и её создания при отсутствии <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def ensure_directory(path, description="", name=""):
    if not os.path.isdir(path):
        os.makedirs(path)
        print(f"❗📁 [{name}] не найден, создан новый каталог: {os.path.abspath(path)}")
    else:
        print(f"📁 [{name}]; {description}: {os.path.abspath(path)}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

input_log_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_log_data'); ensure_directory(input_log_data, "Путь к каталогу с логами исходных файлов", "input_log_data")
print(f"📁 [input_log_data];     Путь к каталогу с историей исходных данных:{input_log_data}"); os.makedirs("tests/fixtures", exist_ok=True)

input_temp_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_temp_data')           # Путь к каталогу с входными временными файлами
print(f"📁 [input_temp_data];    Путь к каталогу с исходными данными:{input_temp_data}")
input_samples_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_samples')     # Путь к каталогу с входными примерами данных
print(f"📁 [input_samples_data]; Путь к каталогу с примерами данных: {input_samples_data}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
print(f"📁 [directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
print(f"📁 [directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"📁 Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"📁 Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

📂 Директория файла:                                      c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\ipynb_files
📂 Директория СубПроекта:                     c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading
📂 Рабочая директория проекта:                             c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
📂 Рабочая директория проекта для доступа к библиотекам:   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📁 [input_log_data]; Путь к каталогу с логами исходных файлов: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\input_data\input_log_data
📁 [input_log_data];     Путь к каталогу с историей исходных данных:c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\input_data\input_log_data
📁 [input_temp_data];    Путь к каталогу с исходным

In [3]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

import csv
import pandas as pd
import json
from clickhouse_driver import Client
import gc
import numpy as np
import glob
from collections import Counter

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                        # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                                   # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                           # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                        # Вывод ДФ
                "df_to_csv",                            # Сохранение ДФ в CSV 
                "CSVLoader",
                "save_data_log_work_file",
                "detect_encoding",
                "time_to_minutes",
                "load_string_list",
                "list_print",
                "move_column"],
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql",
                "get_sql_tab"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 


 ✅ Импорт [dynamic_import_functions.py] успешен.
Импорт из 'yar_sed_general_lib' успешен: ['pd_set_option', 'df_to_csv', 'CSVLoader', 'save_data_log_work_file', 'detect_encoding', 'time_to_minutes', 'load_string_list', 'list_print', 'move_column']
Импорт из 'sql_request_2' успешен: ['pd_read_sql', 'get_sql_tab']

 Импортированные функции и их параметры:
Функция 'pd_set_option' из модуля 'yar_sed_general_lib' ожидает параметры: name_df: str, df: pandas.core.frame.DataFrame, rows: int = 10, columns: int | None = None, min_rows: int | None = None, width: int = 100
Функция 'df_to_csv' из модуля 'yar_sed_general_lib' ожидает параметры: df, csv_file_path
Функция 'CSVLoader' из модуля 'yar_sed_general_lib' ожидает параметры: file_path, delimiter=';', encoding='utf-8', df_name='dataframe'
Функция 'save_data_log_work_file' из модуля 'yar_sed_general_lib' ожидает параметры: df, file_name, directory_data_temp_files, directory_data_log_files
Функция 'detect_encoding' из модуля 'yar_sed_general_li

In [4]:
host='127.0.0.1'
port=9000
user='chbvision'
password='DriavElfimBi'
database='default'

#click_house_client_dict = {'host':'127.0.0.1', 'port':9000, 'user':'chbvision', 'password':'DriavElfimBi', 'database':'default'}

client = Client(host=host, port=port, user=user, password=password, database=database)
print("Connected to ClickHouse", client)
version = client.execute('SELECT version()')
print('ClickHouse version:', version)

Connected to ClickHouse <clickhouse_driver.client.Client object at 0x0000015E218A5DF0>
ClickHouse version: [('25.5.11.15',)]


In [ ]:
# Загружаем CSV с расписанием, что бы составить список символов по которым будем запрашивать котировки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
#df_sessions = pd.read_csv('sessions.csv') # ваш файл

file_path_symbols = Path(directory_data_temp_files)  / "df_sessions_enriched.csv"                       # Файл с торговыми сесcиями
print("latest_symbolTicks_df.csv:", file_path_symbols)
loader = imported["CSVLoader"](file_path_symbols, delimiter=',', encoding='utf-8', df_name='my_dataframe')  # CSVLoader для загрузки данных из файла
df_sessions = loader.load_data()
imported["pd_set_option"]("df_sessions", df_sessions, 5) # Вывод ДФ для проверки

unique_symbols = df_sessions['name_s'].unique()     # Уникальные символы по колонке 'name_s'
imported["list_print"](unique_symbols, "Уникальные символы в df_sessions['name_s']")

latest_symbolTicks_df.csv: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\df_sessions_enriched.csv
✅ Success: [class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\df_sessions_enriched.csv'.

df_sessions  (1,655 строк × 21 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,0_0,1_0,0_1,1_1,0_2,1_2,0_3,1_3,0_4,1_4,0_5,1_5,0_6,1_6,id
0,AUDCAD,AUD / CAD,2.0,Minor,4.0,1,21:00-23:59,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,00:00-20:59,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1654,AEDUSD,AEDUSD,33.0,CFDs - Stocks United States,4.0,10829,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,33.0


📝 [ 1655 ] элементов в списке [ Уникальные символы в df_sessions['name_s'] ] список: ['AUDCAD' 'AUDCHF' 'AUDCNH' 'AUDDKK' 'AUDHUF' 'AUDINR' 'AUDJPY' 'AUDMEX'
 'AUDNOK' 'AUDNZD']...


In [8]:
# Анализируем уже обработанные файлы, и формируем список для обработки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

save_path = os.path.join(directory_data_temp_files, "2026-04-19 Monitor_quotes_according_session") # Директория для сохранения

def get_processed_tickers(path):
    if not os.path.exists(path):
        return []
    
    # Список для найденных тикеров
    processed_tickers = []
    
    # Список зарезервированных имен (те, что мы могли переименовать)
    forbidden_names = {"PRN", "CON", "AUX", "NUL"} # и так далее

    for file in os.listdir(path):
        if file.endswith(".json"):
            # 1. Отрезаем .json
            ticker = file[:-5]
            
            # 2. Если мы добавляли '_' в начале для PRN, CON и т.д. — убираем его
            # Проверяем: начинается с _ И то, что после него (до точки), было в forbidden
            if ticker.startswith("_"):
                base_part = ticker[1:].split('.')[0].upper()
                if base_part in forbidden_names:
                    ticker = ticker[1:]
            
            # 3. Возвращаем оригинальные слеши, если вы их заменяли на '_'
            # ticker = ticker.replace('_', '/') 
            
            processed_tickers.append(ticker)
            
    return processed_tickers

# Получаем список
already_done = get_processed_tickers(save_path)

print(f"Найдено готовых файлов: {len(already_done)};")
print(f"save_path для проверки: {save_path}")
if already_done: imported["list_print"](already_done, "Уже обработанные символы")

# Теперь можно отфильтровать список unique_symbols, чтобы не делать работу дважды
unique_symbols_to_process = [s for s in unique_symbols if s not in already_done]
print(f"Осталось обработать: {len(unique_symbols_to_process)}")

Найдено готовых файлов: 0;
save_path для проверки: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\2026-04-19 Monitor_quotes_according_session
Осталось обработать: 1655


In [9]:
# Создаём базу для хранения результатов по каждому символу, чтобы сохранять после обработки всех блоков одного символа <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
unique_symbols = unique_symbols_to_process
imported["list_print"](unique_symbols, "Символы для обработки")
#unique_symbols = ['EURUSD',]
# 1. ЗАДАЕМ ПРОМЕЖУТОК
start_date = "2026-05-10 00:00:00"
end_date   = "2026-06-05 23:59:59"
save_path = os.path.join(directory_data_temp_files, "monitor")                # Директория для сохранения
print(f"save_path для сохранения: {save_path}")

os.makedirs(save_path, exist_ok=True)                                                   # Создаем папку, если она не существует

BLOCK_SIZE = 100_000       # размер блока строк

full_range = pd.date_range(start=start_date, end=end_date, freq='min')
all_minute_columns = full_range.strftime('%m.%d.%H.%M').tolist()

start_ms = int(pd.to_datetime(start_date).timestamp() * 1000)
end_ms   = int(pd.to_datetime(end_date).timestamp() * 1000)

iter = 0

for sym in unique_symbols:
    current_ts = start_ms
    print(f"⏳ Processing {sym}...")
    
    symbol_results = [] # Временное хранилище блоков для одного символа

    while current_ts < end_ms:
        query = f"SELECT createdTimestampMs FROM default.symbolTicks PREWHERE symbol = '{sym}' WHERE createdTimestampMs >= {current_ts} AND createdTimestampMs <= {end_ms} ORDER BY createdTimestampMs LIMIT {BLOCK_SIZE}"

        try:
            rows = client.execute(query)
        except Exception as e:
            print(f"❌ ERROR: {sym}: {e}")
            break

        if not rows: break

        df_block = pd.DataFrame(rows, columns=['ts'])
        df_block['minute_key'] = pd.to_datetime(df_block['ts'], unit='ms').dt.strftime('%m.%d.%H.%M')
        
        counts = df_block.groupby('minute_key').size()
        symbol_results.append(counts)

        max_ts = df_block['ts'].max()
        if max_ts >= end_ms: break
        current_ts = max_ts + 1
        del df_block; gc.collect()

    # --- СОХРАНЕНИЕ ПОСЛЕ СБОРА ВСЕХ БЛОКОВ СИМВОЛА ---
    if symbol_results:
        # Объединяем все блоки по этому символу в одну серию
        df_sym_total = pd.concat(symbol_results).groupby(level=0).sum()
        # Добавляем недостающие минуты (0) через reindex
        df_sym_total = df_sym_total.reindex(all_minute_columns, fill_value=0)
    else:
        # Если тиков не было вообще, создаем серию из нулей
        df_sym_total = pd.Series(0, index=all_minute_columns)

    # Формируем структуру словаря
    # Ключ: символ, Значение: словарь {минута: количество}
    output_data = {
        sym: df_sym_total.to_dict()
    }

    """В операционной системе Windows слово PRN является зарезервированным системным именем (еще со времен MS-DOS это сокращение для «Printer»).
    Вы не можете создать файл или папку с именем PRN, CON, AUX, NUL, COM1, LPT1 и так далее, даже если там есть расширение.
    Как это исправить? Нам нужно добавить небольшую проверку или модификацию имени файла, чтобы обходить зарезервированные имена Windows.
    Обновите блок формирования имени файла в вашем коде следующим образом:"""
    
    # нужно вынести в отдельный файл
    forbidden_names = {"CON", "PRN", "AUX", "NUL", "COM1", "COM2", "COM3", "COM4",      # Список запрещенных имен Windows
                       "COM5", "COM6", "COM7", "COM8", "COM9", "LPT1", "LPT2", 
                       "LPT3", "LPT4", "LPT5", "LPT6", "LPT7", "LPT8", "LPT9"}
    clean_name = sym.replace('/', '_')                                                  # Имя инструмента для файла
    base_name = clean_name.split('.')[0].upper()                                        # Если имя (до точки) совпадает с запрещенным, добавляем символ подчеркивания
    if base_name in forbidden_names: clean_name = f"_{clean_name}"
    file_name = f"{clean_name}.json"


    # Имя файла (заменяем недопустимые символы в имени, если есть)
    #file_name = f"{sym.replace('/', '_')}.json"
    full_file_path = os.path.join(save_path, file_name)

    # Записываем в JSON
    with open(full_file_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False)
    iter += 1
    print(f"iter {iter}; ✅ Successful Saved: {file_name}")
    
    del symbol_results, df_sym_total, output_data       # Очистка памяти перед следующим инструментом
    gc.collect()

📝 [ 1655 ] элементов в списке [ Символы для обработки ] список: ['AUDCAD', 'AUDCHF', 'AUDCNH', 'AUDDKK', 'AUDHUF', 'AUDINR', 'AUDJPY', 'AUDMEX', 'AUDNOK', 'AUDNZD']...
save_path для сохранения: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\monitor


Error on 127.0.0.1:9000 ping: [WinError 10053] An established connection was aborted by the software in your host machine
Connection was closed, reconnecting.


⏳ Processing AUDCAD...
iter 1; ✅ Successful Saved: AUDCAD.json
⏳ Processing AUDCHF...
iter 2; ✅ Successful Saved: AUDCHF.json
⏳ Processing AUDCNH...
iter 3; ✅ Successful Saved: AUDCNH.json
⏳ Processing AUDDKK...
iter 4; ✅ Successful Saved: AUDDKK.json
⏳ Processing AUDHUF...
iter 5; ✅ Successful Saved: AUDHUF.json
⏳ Processing AUDINR...
iter 6; ✅ Successful Saved: AUDINR.json
⏳ Processing AUDJPY...
iter 7; ✅ Successful Saved: AUDJPY.json
⏳ Processing AUDMEX...
iter 8; ✅ Successful Saved: AUDMEX.json
⏳ Processing AUDNOK...
iter 9; ✅ Successful Saved: AUDNOK.json
⏳ Processing AUDNZD...
iter 10; ✅ Successful Saved: AUDNZD.json
⏳ Processing AUDPLN...
iter 11; ✅ Successful Saved: AUDPLN.json
⏳ Processing AUDRUB...
iter 12; ✅ Successful Saved: AUDRUB.json
⏳ Processing AUDSEK...
iter 13; ✅ Successful Saved: AUDSEK.json
⏳ Processing AUDSGD...
iter 14; ✅ Successful Saved: AUDSGD.json
⏳ Processing AUDTRL...
iter 15; ✅ Successful Saved: AUDTRL.json
⏳ Processing AUDZAR...
iter 16; ✅ Successful Save

In [ ]:
# Нужно доработать для сохранения по сессиям
from datetime import datetime
import re
# --- НАСТРОЙКИ ИНТЕРВАЛА ---
date_start_limit = "2026-04-19"  # С какой даты начинаем (включительно)
date_end_limit   = "2026-04-19"  # По какую дату заканчиваем (включительно)
folder_mask = "Monitor_quotes_according_session"

# Конвертируем границы в объекты даты для сравнения
limit_start = datetime.strptime(date_start_limit, "%Y-%m-%d")
limit_end   = datetime.strptime(date_end_limit, "%Y-%m-%d")

all_dfs = []

# 1. Получаем список всех подкаталогов в directory_data_temp_files
if os.path.exists(directory_data_temp_files):
    subdirs = [d for d in os.listdir(directory_data_temp_files) 
               if os.path.isdir(os.path.join(directory_data_temp_files, d))]
    
    print(f"🔍 Найдено всего папок в temp: {len(subdirs)}")
    
    # 2. Фильтруем папки по маске и дате
    valid_folders = []
    for folder in subdirs:
        if folder.endswith(folder_mask):
            # Вычленяем дату из названия папки
            match = re.search(r'(\d{4}-\d{2}-\d{2})', folder)
            if match:
                folder_date_str = match.group(1)
                folder_date = datetime.strptime(folder_date_str, "%Y-%m-%d")
                
                # Проверяем вхождение в интервал
                if limit_start <= folder_date <= limit_end:
                    valid_folders.append(folder)
    
    print(f"📂 Папок подходит под критерии: {len(valid_folders)}")
    
    # 3. Словарь для хранения данных: { 'EURUSD': Series_с_данными, ... }
    symbols_accumulator = {}

    for folder in sorted(valid_folders):
        save_path = os.path.join(directory_data_temp_files, folder)
        files = glob.glob(os.path.join(save_path, "*.json"))
        print(f"📄 Обработка каталога: 📂 {folder}; 📄 Найдено JSON файлов: {len(files)}")
        
        for f in files:
            try:
                with open(f, 'r', encoding='utf-8') as j:
                    data = json.load(j)
                    if data:
                        sym = list(data.keys())[0]
                        new_data = pd.Series(data[sym], name=sym)
                        
                        if sym in symbols_accumulator:                                                  # Если тикер уже есть, объединяем данные (update). 
                            symbols_accumulator[sym] = symbols_accumulator[sym].combine_first(new_data) # fillna(0) или просто update, чтобы заполнить пропуски за новый день
                        else:
                            symbols_accumulator[sym] = new_data
            except Exception as e: print(f"❌ERROR: Ошибка при чтении файла {f}: {e}")

    if symbols_accumulator:                                                             # 4. Сборка итогового DataFrame из значений словаря
        df_density = pd.DataFrame.from_dict(symbols_accumulator, orient='index')        # Теперь количество строк будет строго равно количеству уникальных тикеров
        print(f"\n✅Success: Итоговый df_density сформирован; 📊 Уникальных тикеров: {len(df_density)}")
    else:
        print("\n❌ERROR: Данные не найдены.")
        df_density = pd.DataFrame()

else:
    print(f"❌ERROR: Базовый каталог temp_files НЕ найден: {directory_data_temp_files}")
    df_density = pd.DataFrame()

imported["save_data_log_work_file"](df_density, "Monitor_quotes_according_session.csv", directory_data_temp_files, directory_data_log_files)
imported["pd_set_option"]("[ df_density ] Monitor_quotes_according_session", df_density, 5)